# The PFS endpoint — two routes to progression-free survival

**NOT FOR CLINICAL USE.** Population/trial-level only; illustrative, unverified parameters. Executed in CI (nbmake).

Onkos computes progression-free survival two legitimate ways that need not agree:

* **statistical** — the parametric PFS survival link, a hazard model keyed on the *week-8* tumor change (the standard route, fit because that is what an early read affords); and
* **mechanistic** — the RECIST time-to-progression read directly off the simulated tumor trajectory (the first time the SLD rises +20% above its running nadir).

The week-8 link is blind to a regrowth tail it never sampled; the mechanism watches the SLD cross progression. For shrink-then-regrow resistance dynamics the two routes **invert the model ranking** — the PFS *route* is a model-selection axis, exactly like the v0.25 OS-metric choice. This is `docs/specs/research/pfs-endpoint.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.response import progression_free_survival, pfs_route_divergence, time_to_progression

ds = onkos.load()
CONTEXTS = ['NSCLC', 'breast', 'CRC', 'HCC', 'melanoma']
print(f'{len(ds)} records | onkos {onkos.__version__}')

## One model, two PFS numbers

The two-population (mechanistic resistance) model shrinks deeply early, then a pre-existing resistant subclone regrows fast. The mechanistic route sees the regrowth; the week-8 link reads only the deep early shrinkage.

In [ ]:
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
pf = progression_free_survival(ds, 'resistance.nsclc_first_line.two_population', context=ctx)
print(f'mechanistic median TTP   = {pf.median_ttp_weeks:.0f} wk  ({pf.ttp_censored_fraction:.0%} censored)')
print(f'statistical median PFS   = {pf.median_pfs_link_weeks:.0f} wk  (week-8 link)')
print(f'progression-free @ {pf.landmark_weeks:.0f} wk = {pf.mechanistic_pfs_rate:.2f}')
print(f'route ratio (mech/stat)  = {pf.route_ratio:.2f}')

## The route inverts the model ranking (NSCLC 1L)

Mechanistically Claret progresses far later than the two-population model; statistically the order flips, because the week-8 link rewards the two-population model's deep early shrinkage. `pfs_route_divergence` counts the model pairs whose mechanistic ranking contradicts their statistical ranking.

In [ ]:
div = pfs_route_divergence(ds, context=ctx, n=300)
print(f"{'model':<46}{'mech TTP':>9}{'stat PFS':>9}{'ratio':>7}")
for r in sorted(div.rows, key=lambda x: -(x['median_ttp_weeks'] or 0)):
    print(f"{r['record_id']:<46}{r['median_ttp_weeks']:>9.0f}{r['median_pfs_link_weeks']:>9.0f}{r['route_ratio']:>7.2f}")
print(f'\nroute-discordant pairs = {div.discordant_pairs}/{div.total_pairs} -> routes_agree = {div.routes_agree}')

## Dataset-wide: not an NSCLC artifact

Every solid-tumor context has both a PFS link (v0.12) and a two-population model (v0.29), so the route inversion reproduces everywhere — the mechanistic route ranks Claret well above the two-population model, the statistical route ranks them level or reversed.

In [ ]:
print(f"{'context':<10}{'discordant':>12}{'Claret mech/stat':>20}{'two-pop mech/stat':>20}")
for tt in CONTEXTS:
    c = {'tumor_type': tt, 'line': 'first'}
    div = pfs_route_divergence(ds, context=c, n=200)
    cl = progression_free_survival(ds, 'resistance.claret_2009.tgi', context=c, n=200)
    tp_id = next(r.id for r in ds if r.id.endswith('two_population')
                 and r.derivation_context and r.derivation_context.tumor_type == tt)
    tp = progression_free_survival(ds, tp_id, context=c, n=200)
    print(f"{tt:<10}{f'{div.discordant_pairs}/{div.total_pairs}':>12}"
          f"{f'{cl.median_ttp_weeks:.0f} / {cl.median_pfs_link_weeks:.0f}':>20}"
          f"{f'{tp.median_ttp_weeks:.0f} / {tp.median_pfs_link_weeks:.0f}':>20}")

## Guardrails ride through

Worst-input-wins governs the PFS endpoint exactly as it governs OS/ORR: transporting Claret (NSCLC-validated) onto melanoma floors the tier to **D**, and the clinical-use prohibition is on every payload. The mechanistic and running-nadir progression rule is closed-form testable.

In [ ]:
# Out-of-context transport floors the tier; the payload carries the prohibition.
oos = progression_free_survival(ds, 'resistance.claret_2009.tgi',
                                context={'tumor_type': 'melanoma', 'line': 'first'}, n=60)
print('transported tier =', oos.tier, '| clinical use =', oos.to_dict()['onkos:clinicalUse'][:24], '...')

# Closed-form: progression is measured against the RUNNING NADIR, not baseline.
t = np.linspace(0, 104, 209)
v = np.concatenate([np.linspace(100, 40, 105), np.linspace(40, 60, 104)])  # nadir 40 -> +50%
print('mechanistic TTP (closed form) =', round(time_to_progression(t, v), 1), 'wk (progresses below baseline)')

## The takeaway

PFS is **not a single number** — it carries a route choice (statistical hazard link vs mechanistic RECIST progression), that choice is a model-selection axis, and for the resistance dynamics that gate accelerated approvals the two routes invert the model ranking in every solid-tumor context. Onkos reports the disagreement and privileges neither route; CI enforces it (`tests/test_pfs_routes.py`).